# ShieldShot Training Pipeline

Run this notebook on RunPod after cloning the repo and installing deps:
```bash
cd /workspace
git clone https://github.com/varmabudharaju/ShieldShot.git shieldshot
cd shieldshot
pip install -e ".[dev]"
pip install lpips gdown
```

## 1. Setup & GPU Check

In [ ]:
import os
os.chdir("/workspace/shieldshot")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Download FFHQ Dataset

In [ ]:
import subprocess
from pathlib import Path

DATA_DIR = Path("/workspace/data")
FFHQ_DIR = DATA_DIR / "ffhq"
MODELS_DIR = Path("/workspace/shieldshot/trained_models")

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Check if FFHQ already downloaded
if FFHQ_DIR.exists() and len(list(FFHQ_DIR.glob("*.png"))) > 1000:
    print(f"FFHQ already exists: {len(list(FFHQ_DIR.glob('*.png')))} images")
else:
    print("Downloading FFHQ thumbnails (~1.8GB)...")
    FFHQ_DIR.mkdir(parents=True, exist_ok=True)
    !gdown --fuzzy "https://drive.google.com/uc?id=1SH5a4M5IpEGZINmE_4FnmMKjcRmYEbak" -O {DATA_DIR}/ffhq_thumbs.zip
    !cd {DATA_DIR} && unzip -q ffhq_thumbs.zip -d ffhq_temp
    !find {DATA_DIR}/ffhq_temp -name "*.png" -exec mv {{}} {FFHQ_DIR}/ \;
    !rm -rf {DATA_DIR}/ffhq_temp {DATA_DIR}/ffhq_thumbs.zip
    print(f"FFHQ ready: {len(list(FFHQ_DIR.glob('*.png')))} images")

## 3. Train Watermark Encoder/Decoder

**Loss:** BCE (payload recovery) + SSIM + LPIPS (visual quality) + high-freq penalty (adversarial compatibility)

**~2 hours on RTX 4090**

In [ ]:
!python3 -m train.train_watermark \
    --data-dir {FFHQ_DIR} \
    --output-dir {MODELS_DIR} \
    --epochs 100 \
    --batch-size 32 \
    --image-size 256 \
    --lr 1e-3 \
    --w-bce 1.0 \
    --w-ssim 0.3 \
    --w-lpips 0.7 \
    --w-compat 0.1 \
    --num-workers 4 \
    --save-every 10 \
    --resume

## 4. Train Perturbation Generator

**Loss:** Feature distortion (all 5 target models) + LPIPS quality + JPEG compression robustness

**~2-3 hours on RTX 4090**

In [ ]:
# Create 5000-image subset for generator training
import random

GEN_SUBSET_DIR = DATA_DIR / "ffhq_gen_subset"
GEN_SUBSET_DIR.mkdir(parents=True, exist_ok=True)

if len(list(GEN_SUBSET_DIR.glob("*"))) < 5000:
    all_images = sorted(FFHQ_DIR.glob("*.png"))
    subset = random.sample(all_images, min(5000, len(all_images)))
    for img in subset:
        link = GEN_SUBSET_DIR / img.name
        if not link.exists():
            link.symlink_to(img)
    print(f"Subset ready: {len(list(GEN_SUBSET_DIR.glob('*')))} images")
else:
    print(f"Subset already exists: {len(list(GEN_SUBSET_DIR.glob('*')))} images")

In [ ]:
!python3 -m train.train_generator \
    --data-dir {GEN_SUBSET_DIR} \
    --epochs 50 \
    --batch-size 8 \
    --image-size 256 \
    --lr 1e-4 \
    --w-distortion 1.0 \
    --w-quality 0.5 \
    --w-compression 0.3 \
    --num-workers 4 \
    --save-every 10 \
    --resume \
    --output {MODELS_DIR}/generator.pt

## 5. Validation

In [ ]:
# Check model files
for f in ["encoder.pt", "decoder.pt", "generator.pt"]:
    path = MODELS_DIR / f
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"  OK: {f} ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING: {f}")

In [ ]:
# Watermark round-trip test
from shieldshot.watermark.encoder import WatermarkEncoder
from shieldshot.watermark.decoder import WatermarkDecoder
from shieldshot.watermark.payload import encode_payload, decode_payload, PAYLOAD_BITS
from shieldshot.utils.quality import compute_ssim, compute_lpips

device = "cuda" if torch.cuda.is_available() else "cpu"

encoder = WatermarkEncoder(payload_bits=PAYLOAD_BITS).to(device)
decoder = WatermarkDecoder(payload_bits=PAYLOAD_BITS).to(device)
encoder.load_state_dict(torch.load(MODELS_DIR / "encoder.pt", weights_only=True))
decoder.load_state_dict(torch.load(MODELS_DIR / "decoder.pt", weights_only=True))
encoder.eval()
decoder.eval()

test_img = torch.randn(1, 3, 256, 256).clamp(0, 1).to(device)
payload = encode_payload(user_id="test", timestamp=1234567890)
payload_tensor = torch.tensor(payload, dtype=torch.float32).unsqueeze(0).to(device)

with torch.no_grad():
    watermarked = encoder(test_img, payload_tensor).clamp(0, 1)
    logits = decoder(watermarked)
    predicted_bits = (logits > 0).int().squeeze(0).tolist()

result = decode_payload(predicted_bits)
ssim_val = compute_ssim(test_img, watermarked)
lpips_val = compute_lpips(test_img, watermarked)
bit_acc = sum(a == b for a, b in zip(payload, predicted_bits)) / len(payload)

print(f"Watermark SSIM:       {ssim_val:.4f}  (target: >0.95)")
print(f"Watermark LPIPS:      {lpips_val:.4f}  (target: <0.05)")
print(f"Payload valid:        {result['valid']}")
print(f"Bit accuracy:         {bit_acc:.2%}")

In [ ]:
# Generator test
from shieldshot.perturb.generator import PerturbationGenerator

gen = PerturbationGenerator().to(device)
gen.load_state_dict(torch.load(MODELS_DIR / "generator.pt", weights_only=True))
gen.eval()

face = torch.randn(1, 3, 256, 256).clamp(0, 1).to(device)
with torch.no_grad():
    perturbed = gen(face)

diff = (perturbed - face).abs().max().item()
gen_ssim = compute_ssim(face, perturbed)
gen_lpips = compute_lpips(face, perturbed)

print(f"Max perturbation:     {diff:.4f}  (budget: {8/255:.4f})")
print(f"Generator SSIM:       {gen_ssim:.4f}  (target: >0.95)")
print(f"Generator LPIPS:      {gen_lpips:.4f}  (target: <0.05)")

## 6. Visual Comparison

In [ ]:
import matplotlib.pyplot as plt
from shieldshot.utils.image import load_image, to_tensor, to_pil

# Pick a real FFHQ image
sample_path = sorted(FFHQ_DIR.glob("*.png"))[0]
sample_img = load_image(str(sample_path))
sample_tensor = to_tensor(sample_img).to(device)

# Resize to 256x256 for the models
sample_tensor = torch.nn.functional.interpolate(sample_tensor, size=(256, 256), mode="bilinear", align_corners=False)

with torch.no_grad():
    # Watermark
    wm_payload = encode_payload(user_id="demo", timestamp=1234567890)
    wm_tensor = torch.tensor(wm_payload, dtype=torch.float32).unsqueeze(0).to(device)
    watermarked = encoder(sample_tensor, wm_tensor).clamp(0, 1)
    
    # Perturbation
    perturbed = gen(sample_tensor)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

def show(ax, tensor, title):
    img = tensor.squeeze(0).permute(1, 2, 0).cpu().clamp(0, 1).numpy()
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis("off")

# Row 1: Watermark
show(axes[0, 0], sample_tensor, "Original")
show(axes[0, 1], watermarked, f"Watermarked\nSSIM={compute_ssim(sample_tensor, watermarked):.4f}")
diff_wm = (watermarked - sample_tensor).abs() * 10  # amplify for visibility
show(axes[0, 2], diff_wm, "Watermark Residual (10x)")

# Row 2: Perturbation
show(axes[1, 0], sample_tensor, "Original")
show(axes[1, 1], perturbed, f"Perturbed\nSSIM={compute_ssim(sample_tensor, perturbed):.4f}")
diff_p = (perturbed - sample_tensor).abs() * 10
show(axes[1, 2], diff_p, "Perturbation Residual (10x)")

plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to training_results.png")

## 7. Download Models

From your local machine:
```bash
mkdir -p ~/.shieldshot/models
scp runpod:/workspace/shieldshot/trained_models/*.pt ~/.shieldshot/models/
```